In [1]:
import sys
import os
sys.path.append('/root/capsule/code/beh_ephys_analysis')
from harp.clock import decode_harp_clock, align_timestamps_to_anchor_points
from open_ephys.analysis import Session
import datetime
from aind_ephys_rig_qc.temporal_alignment import search_harp_line
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
from pynwb import NWBFile, TimeSeries, NWBHDF5IO
from scipy.io import loadmat
from scipy.stats import zscore
import ast
from utils.plot_utils import combine_pdf_big

from open_ephys.analysis import Session
from pathlib import Path
import glob

import json
import seaborn as sns
from PyPDF2 import PdfMerger
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import re
from aind_dynamic_foraging_basic_analysis.plot.plot_foraging_session import plot_foraging_session
from aind_dynamic_foraging_data_utils.nwb_utils import load_nwb_from_filename
from hdmf_zarr.nwb import NWBZarrIO
from utils.beh_functions import session_dirs, parseSessionID, load_model_dv, makeSessionDF, get_session_tbl, get_unit_tbl, get_history_from_nwb
from utils.ephys_functions import*
from utils.opto_utils import opto_metrics, load_opto_sig
import pandas as pd
import pickle
import scipy.stats as stats
from joblib import Parallel, delayed
from multiprocessing import Pool
from functools import partial
import time
import spikeinterface as si
import shutil 
import seaborn as sns
import math
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import zscore
%matplotlib inline

In [2]:
# Make combined session-unit table
dfs = [pd.read_csv('/root/capsule/code/data_management/session_assets.csv')]
df = pd.concat(dfs)
# df = df[-10:]
session_ids = df['session_id'].values
behs = df['behavior'].values
exclude = ['ecephys_717120_2024-03-06_12-23-53', 'ecephys_713854_2024-03-08_14-54-25', 'ecephys_713854_2024-03-08_16-20-33', 'behavior_754897_2025-03-15_11-32-18']
session_ids, behs = zip(*[
    (session, beh)
    for session, beh in zip(session_ids, behs)
    if isinstance(session, str) and session not in exclude
])
session_ids = list(session_ids)
behs = list(behs)

In [3]:
# loop through all sessions
# count number of units pass qc and opto qc
# check if exist in longer than 100 sessions 
# check if exist in pre/post stimulation
# check if exist in anti-dromic stimulation
all_sessions = []
all_units = []
all_qc_pass_units = [] # default qc pass
all_real_units = [] # not noise, not artifact
all_tagged_units = []
in_behavior = []
trial_count = []
opto_tagging_pre = []
opto_tagging_post = []
anti = []
probes = []
all_p_max = []
all_p_mean = []
all_lat_max_p = []  # latency of max p response
isi_v = []
all_eu = []
all_corr = []
all_amp = []
all_peak = []
all_wf = []
all_wf_aligned = []
all_wf_raw = []
all_wf_2d = []
all_wf_2d_raw = []
all_peak_raw = []
all_amp_raw = []
y_loc = []
in_df = []
rec_side = []
top = []
bottom = []
snr = []
all_tag_loc = []
all_fr = []
all_sig_counts = []
all_decoder = []
all_peak_channel_ind = []
resp_p_all_conditions = []
mean_p_all_conditions = []
resp_lat_all_conditions = []
corr_all_conditions = []
eu_all_conditions = []
sig_counts_all_conditions = []
all_x = []
all_y = []
all_z = []

# p_resp_thresh = 0.5
# lat_resp_thresh = 0.02


In [4]:
target = 'soma'
for session, beh in zip(session_ids, behs):
    session_dir = session_dirs(session)
    if 'ZS' in session:
        if (not os.path.exists(session_dir['nwb_dir_raw'])) or (get_unit_tbl(session, 'curated') is None):
            print(f'Skipping {session} due to no neuron data')
            continue
    if session_dir['opto_dir_raw'] is not None:
        print(f'Processing {session}')
        data_type = 'raw'

        try:
            unit_tbl = get_unit_tbl(session, data_type)
        except (EOFError, pickle.UnpicklingError) as e:
            print(f"{session}: bad pickle ({type(e).__name__}), skip")
            continue

        if unit_tbl is None:
            print(f"{session}: no unit table, skip")
            continue

        required_cols = ["unit_id", "default_qc"]
        missing_cols = [c for c in required_cols if c not in unit_tbl.columns]

        if missing_cols:
            print(f"{session}: missing columns {missing_cols}, got columns {unit_tbl.columns.tolist()}, skip")
            continue

        
        # if session_dir['curated_dir_curated'] is not None:
        #     data_type = 'curated'
        # elif session_dir['curated_dir_raw'] is not None:
        #     data_type = 'raw'
        # else:
        #     continue
        qm_file = os.path.join(session_dir['processed_dir'], )
        unit_tbl = get_unit_tbl(session, data_type)
        opto_metrics_session = opto_metrics(session, data_type=data_type)
        # unit_nwb = load_nwb_from_filename(session_dir['nwb_dir_curated'])
        # unit_temps = unit_nwb.units[:]['waveform_mean'].values
        # peak_C = [np.argmax(np.ptp(curr_wf, axis=0)) if isinstance(curr_wf, np.ndarray) else None for curr_wf in unit_temps]  # peak channel index
        # print(np.shape(unit_temps[0]))
        all_units.extend(unit_tbl['unit_id'].tolist())
        all_qc_pass_units.extend(unit_tbl['default_qc'].tolist())  # default qc pass
        # all_real_units.extend(unit_tbl['real_unit'].tolist())
        all_tagged_units.extend(unit_tbl['tagged_loc'].tolist())  # tagged location (e.g. 'soma', 'axon', 'unspecified')
        all_sessions.extend([session]*len(unit_tbl))
        # all_peak_channel_ind.extend(peak_C)  # peak channel index
        if 'p_max' not in unit_tbl.columns:
            all_p_max.extend(unit_tbl['p_max_x'].tolist())
            all_p_mean.extend(unit_tbl['p_mean_x'].tolist())
            all_lat_max_p.extend(unit_tbl['lat_max_p_x'].tolist())
            all_eu.extend(unit_tbl['euc_max_p_x'].tolist())
            all_corr.extend(unit_tbl['corr_max_p_x'].tolist())
            peaks = unit_tbl['peak_x'].values
            amp = unit_tbl['amp_x'].values
        else: 
            all_p_max.extend(unit_tbl['p_max'].tolist())
            all_p_mean.extend(unit_tbl['p_mean'].tolist())  
            all_lat_max_p.extend(unit_tbl['lat_max_p'].tolist()) 
            all_eu.extend(unit_tbl['euc_max_p'].tolist())
            all_corr.extend(unit_tbl['corr_max_p'].tolist()) 
            peaks = unit_tbl['peak'].values
            amp = unit_tbl['amp'].values
        
        if 'x_ccf' in unit_tbl.columns:
            all_x.extend(unit_tbl['x_ccf'].tolist())
            all_y.extend(unit_tbl['y_ccf'].tolist())
            all_z.extend(unit_tbl['z_ccf'].tolist())
        else:
            all_x.extend([np.nan]*len(unit_tbl))
            all_y.extend([np.nan]*len(unit_tbl))
            all_z.extend([np.nan]*len(unit_tbl))
     
        isi_v.extend(unit_tbl['isi_violations_ratio'].tolist())  # ISI violations 
        snr.extend(unit_tbl['snr'].tolist())  # signal-to-noise ratio
        y_loc.extend(unit_tbl['y_loc'].tolist())  # y location of the unit
        all_fr.extend(unit_tbl['firing_rate'].tolist())  # firing rate
        all_decoder.extend(unit_tbl['decoder_label'].tolist())  # decoder value
        if 'tagged_loc' in unit_tbl.columns:
            all_tag_loc.extend(unit_tbl['tagged_loc'].tolist())
        else:
            all_tag_loc.extend([np.nan]*len(unit_tbl))
        if 'peak_wf_opt' in unit_tbl.columns:
            wf_opt = [wf_opt_unit if isinstance(wf_opt_unit, np.ndarray) else wf_unit for wf_opt_unit, wf_unit in zip(unit_tbl['peak_wf_opt'], unit_tbl['peak_wf'])]  # peak waveform
            wf_opt_aligned = [wf_opt_unit if isinstance(wf_opt_unit, np.ndarray) else wf_unit for wf_opt_unit, wf_unit in zip(unit_tbl['peak_wf_opt_aligned'], unit_tbl['peak_wf_aligned'])]  # peak waveform aligned
            wf_opt_2d = [wf_opt_unit if isinstance(wf_opt_unit, np.ndarray) else wf_unit for wf_opt_unit, wf_unit in zip(unit_tbl['mat_wf_opt'], unit_tbl['wf_2d'])]  # peak waveform 2D
        else:
            wf_opt = unit_tbl['peak_wf'].values.tolist()
            wf_opt_aligned = unit_tbl['peak_wf_aligned'].values.tolist()
            wf_opt_2d = unit_tbl['wf_2d'].values.tolist()

        amp_opt = [
                        np.max(wf_opt_curr) - np.min(wf_opt_curr) if isinstance(wf_opt_curr, np.ndarray) else curr_amp_unit
                        for wf_opt_curr, curr_amp_unit in zip(wf_opt, amp)
                    ]   # amplitude of optimized waveforms
        if 'amplitude_opt' in unit_tbl.columns:
            peak_opt = [
                            curr_peak_opt_unit if not np.isnan(curr_peak_opt_unit) else curr_peak_unit
                            for curr_peak_opt_unit, curr_peak_unit in zip(list(unit_tbl['amplitude_opt'].values), list(peaks))
                        ]
        else:
            peak_opt = list(peaks)
        
        if 'peak_waveform_raw_aligned' in unit_tbl.columns:
            wf_raw = unit_tbl['peak_waveform_raw_fake_aligned'].values.tolist()
            wf_2d_raw = unit_tbl['mat_wf_raw_fake'].values.tolist()
            peak_raw = unit_tbl['peak_raw_fake'].values.tolist()
            peak_raw = [curr_peak_raw-curr_wf[0] if (curr_peak_raw is not None) and (~np.isnan(curr_peak_raw)) else None for curr_peak_raw, curr_wf in zip(peak_raw, wf_raw)]
            amp_raw = unit_tbl['amplitude_raw_fake'].values.tolist()
        else:
            wf_raw = [None]*len(unit_tbl)
            wf_2d_raw = [None]*len(unit_tbl)
            peak_raw = [None]*len(unit_tbl)
            amp_raw = [None]*len(unit_tbl)
        
        all_amp.extend(amp_opt)  # amplitude 
        all_peak.extend(peak_opt)  # peak opto response
        all_wf.extend(wf_opt)  # waveform 
        all_wf_aligned.extend(wf_opt_aligned)  # waveform of max amp
        all_wf_2d.extend(wf_opt_2d)  # waveform of max amp
        all_wf_raw.extend(wf_raw)  # raw waveform
        all_wf_2d_raw.extend(wf_2d_raw)  # raw waveform 2d
        all_peak_raw.extend(peak_raw)  # raw peak
        all_amp_raw.extend(amp_raw)  # raw amplitude
        
        top.extend(unit_tbl['DRN_range_top'].tolist())
        bottom.extend(unit_tbl['DRN_range_bottom'].tolist())

        # session and opto information
        session_df = get_session_tbl(session)
        session_opto_sig = load_opto_sig(session, data_type=data_type)
        # with open(os.path.join(session_dir[f'opto_dir_{data_type}'], f'{session}_opto_info_{target}.json')) as f:
        #     opto_info = json.load(f)

        if not session_dir['aniID'].startswith('ZS'):
            opto_df = pd.read_csv(os.path.join(session_dir[f'opto_dir_{data_type}'], f'{session}_opto_session_{target}.csv'))
            if len(opto_df[opto_df['pre_post'] == 'pre'])>0:
                pre_end = np.max(opto_df[opto_df['pre_post'] == 'pre']['time'].values)
            else:
                pre_end = np.nan
        
            if len(opto_df[opto_df['pre_post'] == 'post'])>0:
                post_start = np.min(opto_df[opto_df['pre_post'] == 'post']['time'].values)
                post_end = np.max(opto_df[opto_df['pre_post'] == 'post']['time'].values)
            else:
                post_start = np.nan
                post_end = np.nan
        
        for unit_id in unit_tbl['unit_id'].values:
            # append opto_metrics
            spike_times = unit_tbl[unit_tbl['unit_id'] == unit_id]['spike_times'].values[0]
            unit_opto = opto_metrics_session.load_unit(unit_id)
            unit_opto_sig = session_opto_sig.load_unit(unit_id)

            if unit_opto_sig is not None:
                curr_max_count = unit_opto_sig['p_sig_count'].max()
            else:
                curr_max_count = np.nan
            
            curr_p_resp_all = unit_opto['resp_p_bl'].values
            curr_lat_resp_all = unit_opto['resp_lat'].values
            curr_p_mean_all = unit_opto['mean_p'].values
            curr_eu_all = unit_opto['euclidean_norm'].values
            curr_corr_all = unit_opto['correlation'].values
            unit_opto['sig_num'] = np.full(len(unit_opto), np.nan)
            if unit_opto_sig is not None:
                if not session_dir['aniID'].startswith('ZS'):
                    for cond_ind, row in unit_opto.iterrows():
                        filter = (unit_opto_sig['power'] == row['powers']) & (unit_opto_sig['site'] == row['sites'])
                        if len(unit_opto_sig['pre_post'].unique()) > 1:
                            filter &= (unit_opto_sig['pre_post'] == row['stim_times'])
                        curr_sig_rows = unit_opto_sig[filter]
                        if len(curr_sig_rows) == 1:
                            unit_opto.loc[cond_ind, 'sig_num'] = curr_sig_rows['p_sig_count'].values[0]
                        elif len(curr_sig_rows) > 1:
                            print(f'Warning: multiple opto sig rows for unit {unit_id} in session {session} with condition {row["powers"]} and pre_post {row["stim_times"]}')
                            unit_opto.loc[cond_ind, 'sig_num'] = curr_sig_rows['p_sig_count'].values[0]
                        else:
                            print(f'Warning: no opto sig rows for unit {unit_id} in session {session} with condition {row["powers"]} and pre_post {row["stim_times"]}')
                            unit_opto.loc[cond_ind, 'sig_num'] = np.nan
                else:
                    unit_opto['sig_num'] = unit_opto_sig['p_sig_count'].values[0]
            
            curr_sig_num_all = unit_opto['sig_num'].values

            resp_p_all_conditions.append(curr_p_resp_all)
            resp_lat_all_conditions.append(curr_lat_resp_all)
            corr_all_conditions.append(curr_corr_all)
            eu_all_conditions.append(curr_eu_all)
            mean_p_all_conditions.append(curr_p_mean_all)
            sig_counts_all_conditions.append(curr_sig_num_all)
            all_sig_counts.append(curr_max_count)


            if session_dir['aniID'].startswith('ZS'):
                curr_anti_opto = False
                curr_pre_opto = False
                curr_post_opto = True
            else:
                curr_anti_opto = True
                curr_pre_opto = True
                curr_post_opto = True  
            if session_df is not None:
                curr_in_beh = True
            else:
                curr_in_beh = False
                curr_trial_count = 0
            unit = unit_tbl[unit_tbl['unit_id'] == unit_id]
            unit_drift = load_drift(session, unit_id, data_type=data_type)
            
            if session_df is not None:
                go_cue_times = session_df['goCue_start_time']
                if unit_drift is not None:
                    if unit_drift['ephys_cut'][0] is not None:
                        if unit_drift['ephys_cut'][0] > pre_end - 2*60:
                            curr_pre_opto = False 
                        go_cue_times = go_cue_times[go_cue_times >= unit_drift['ephys_cut'][0]]         
                    if unit_drift['ephys_cut'][1] is not None:
                        if unit_drift['ephys_cut'][1] < post_start + 2*60:
                            curr_post_opto = False
                        if unit_drift['ephys_cut'][1] < post_end + 2*60:
                            curr_anti_opto = False
                        go_cue_times = go_cue_times[go_cue_times <= unit_drift['ephys_cut'][1]]
                if len(go_cue_times) < 100: 
                    curr_in_beh = False
                curr_trial_count = len(go_cue_times)
            opto_tagging_pre.append(curr_pre_opto)
            opto_tagging_post.append(curr_post_opto)
            anti.append(curr_anti_opto)
            trial_count.append(curr_trial_count)
            in_behavior.append(curr_in_beh)
        probes.extend([df[df['session_id']==session]['probe'].values[0]] * len(unit_tbl))
        rec_side.extend([df[df['session_id']==session]['side'].values[0]]*len(unit_tbl))  # recording side
        in_df.extend([beh] * len(unit_tbl))  # store session df for each unit


Processing behavior_792292_2025-08-13_14-47-17
Processing behavior_792292_2025-08-15_14-15-47
Processing behavior_791174_2025-09-05_14-06-53


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
hdmf-common - cached version: 1.9.0, loaded version: 1.8.0
hdmf-experimental - cached version: 0.6.0, loaded version: 0.5.0
Please update to the latest package versions.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Processing behavior_814511_2025-10-16_13-44-32
Processing behavior_814515_2025-10-22_13-03-21
Processing behavior_814515_2025-10-23_14-41-04
Processing behavior_814515_2025-10-24_13-22-07
behavior_814515_2025-10-24_13-22-07: missing columns ['unit_id'], got columns ['default_qc'], skip
Processing behavior_810888_2025-10-28_11-51-06
Processing behavior_810888_2025-10-31_12-39-56
Processing behavior_808655_2025-09-16_10-53-22
Processing behavior_808655_2025-09-18_13-56-51
Processing behavior_808650_2025-09-23_14-49-57
Processing behavior_808650_2025-09-24_12-38-00
Processing behavior_808650_2025-09-25_13-35-32
Processing behavior_808650_2025-09-26_11-51-57
Processing behavior_796387_2025-09-09_13-37-29
behavior_796387_2025-09-09_13-37-29: missing columns ['unit_id'], got columns ['default_qc'], skip
Processing behavior_792288_2025-08-26_11-41-18
Processing behavior_792288_2025-08-27_12-09-38
Processing behavior_826160_2025-12-19_11-14-24
Processing behavior_826159_2026-01-22_12-58-47
Pro

In [5]:
combined_tagged_units = pd.DataFrame({'session': all_sessions,
                                        'unit': all_units,
                                        'qc_pass': all_qc_pass_units,
                                        'opto_tagged': all_tagged_units,
                                        'opto_tagging_pre': opto_tagging_pre,
                                        'opto_tagging_post': opto_tagging_post,
                                        'anti': anti,
                                        'in_df': in_behavior,
                                        'trial_count': trial_count,
                                        'p_max': all_p_max,
                                        'sig_counts': all_sig_counts,
                                        'p_mean': all_p_mean,
                                        'lat_max_p': all_lat_max_p,
                                        'isi_violations': isi_v,
                                        'snr': snr,
                                        'eu': all_eu,
                                        'corr': all_corr,
                                        'amp': all_amp,
                                        'amp_raw': all_amp_raw,
                                        'peak': all_peak,
                                        'peak_raw': all_peak_raw,
                                        'wf': all_wf,
                                        'wf_raw': all_wf_raw,
                                        'wf_aligned': all_wf_aligned,
                                        'wf_2d': all_wf_2d,
                                        'wf_2d_raw': all_wf_2d_raw,
                                        'probe': probes,
                                        'y_loc': y_loc, 
                                        'rec_side': rec_side,
                                        'top': top,
                                        'bottom': bottom,
                                        'tag_loc': all_tag_loc,
                                        'fr': all_fr,
                                        'decoder': all_decoder,
                                        #'peak_channel_ind': all_peak_channel_ind,
                                        'all_p_max': resp_p_all_conditions,
                                        'all_p_mean': mean_p_all_conditions,
                                        'all_lat_max_p': resp_lat_all_conditions,
                                        'all_corr': corr_all_conditions,
                                        'all_eu': eu_all_conditions,
                                        'all_sig_counts': sig_counts_all_conditions,
                                        'x_ccf': all_x,
                                        'y_ccf': all_y,
                                        'z_ccf': all_z
                                        }
)


In [19]:
# save dataframe in combined folder
with open(os.path.join('/root/capsule/scratch/combined/combine_unit_tbl', 'combined_unit_tbl.pkl'), 'wb') as f:
    pickle.dump(combined_tagged_units, f)

In [6]:
combined_tagged_units

,session,unit,qc_pass,opto_tagged,opto_tagging_pre,opto_tagging_post,anti,in_df,trial_count,p_max,...,decoder,all_p_max,all_p_mean,all_lat_max_p,all_corr,all_eu,all_sig_counts,x_ccf,y_ccf,z_ccf
0,behavior_792292_2025-08-13_14-47-17,49,True,False,True,True,True,True,282,0.319298,...,mua,[0.31929806067505284],[0.08261550024808828],[0.008790497854351997],[0.25395479750553934],[2.5344160310346235],[1.0],NaN,NaN,NaN
1,behavior_792292_2025-08-13_14-47-17,50,True,False,True,True,True,True,282,0.309539,...,mua,[0.30953908378458217],[-0.01165373711753763],[0.015288160368800163],[0.6558823267921994],[0.9381531991622085],[1.0],NaN,NaN,NaN
2,behavior_792292_2025-08-13_14-47-17,73,True,False,True,True,True,True,282,0.308613,...,sua,[0.30861312115371026],[-0.04923663219961551],[0.011897732503712177],[0.8697839593135852],[0.3884542646608028],[0.0],NaN,NaN,NaN
3,behavior_792292_2025-08-13_14-47-17,82,True,False,True,True,True,True,282,0.350310,...,mua,[0.35030989586081],[0.11499679634332856],[0.00815828720277006],[0.03230200540696425],[2.047074869115651],[0.0],NaN,NaN,NaN
4,behavior_792292_2025-08-13_14-47-17,136,True,False,True,True,True,True,282,0.309399,...,mua,[0.3093994240843326],[0.15973818937820505],[0.009892954677343368],[0.34029135748905565],[3.463190138520919],[1.0],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,behavior_835451_2026-02-27_14-19-03,115,True,True,True,True,True,True,425,0.779462,...,sua,"[0.7794618821460876, 0.5794618821460876, 0.679...","[0.6894618821460876, 0.6894618821460876, 0.689...","[0.01037556743249297, 0.012627743883058429, 0....","[0.9946185493101667, 0.9946185493101667, 0.994...","[0.0964096834549662, 0.0964096834549662, 0.096...","[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",NaN,NaN,NaN
301,behavior_835451_2026-02-27_14-19-03,132,True,True,True,True,True,True,425,0.711678,...,sua,"[0.41458753342097915, 0.5145875334209791, 0.31...","[0.3845875334209791, 0.3845875334209791, 0.384...","[0.011640628799796104, 0.01334784081315293, 0....","[0.9923696543613838, 0.9923696543613838, 0.992...","[0.09753764492136105, 0.09753764492136105, 0.0...","[3.0, 3.0, 3.0, 3.0, 3.0, 5.0, 5.0, 5.0, 5.0, ...",NaN,NaN,NaN
302,behavior_835451_2026-02-27_14-19-03,133,True,False,True,True,True,True,425,0.706108,...,mua,"[0.32595424686724794, 0.32595424686724794, 0.3...","[0.125954246867248, 0.255954246867248, 0.25595...","[0.014961576355355126, 0.011525146131004606, 0...","[0.9556103695720068, 0.9814869777287322, 0.981...","[0.2678034728556756, 0.20678688458382585, 0.20...","[1.0, 3.0, 3.0, 5.0, 5.0, 5.0, 5.0, 5.0, 4.0, ...",NaN,NaN,NaN
303,behavior_835451_2026-02-27_14-19-03,189,True,False,True,True,True,True,425,0.371789,...,sua,[0.37178873201944285],[0.13138609011942753],[0.015365460515022277],[0.9699065953024886],[0.18889079430126013],[3.0],NaN,NaN,NaN


In [9]:
unit_tbl = get_unit_tbl('behavior_754897_2025-03-13_11-20-42', 'curated')

In [10]:
unit_tbl.columns

Index(['unit_id', 'bl_max_p', 'p_max', 'p_mean', 'lat_max_p', 'lat_mean',
       'euc_max_p', 'corr_max_p', 'opto_pass', 'amp', 'peak', 'real_unit',
       'y_loc', 'pass_count', 'spike_times', 'ks_unit_id',
       'isi_violations_ratio', 'firing_rate', 'presence_ratio',
       'amplitude_cutoff', 'decoder_label', 'depth', 'snr', 'waveform_mean',
       'waveform_sd', 'default_qc', 'peak_wf', 'peak_wf_aligned', 'wf_2d',
       'LC_range_top', 'LC_range_bottom', 'tagged_loc', 'tagged',
       'amplitude_opt', 'peak_wf_opt', 'mat_wf_opt', 'peak_wf_opt_aligned',
       'mat_wf_raw', 'mat_wf_raw_aligned', 'peak_waveform_raw',
       'peak_waveform_raw_aligned', 'amplitude_raw', 'peak_raw',
       'mat_wf_raw_fake', 'mat_wf_raw_fake_aligned', 'peak_waveform_raw_fake',
       'peak_waveform_raw_fake_aligned', 'amplitude_raw_fake', 'peak_raw_fake',
       'loc_along_probe_x', 'x_ccf', 'y_ccf', 'z_ccf', 'loc_along_probe_y'],
      dtype='object')